# Búsqueda de configuración (coordenadas)

Búsqueda por coordenadas con el modelo real (`qwen17b`, T4). Mejora una ranura a la vez,
solo acepta si la precisión sube, y repite desde varios arranques al azar.

**Uso:** GPU T4. Ejecute las celdas en orden. Si Colab se desconecta, vuelva a ejecutar todo:
el caché en Drive y las semillas fijas hacen que lo ya generado no se repita.

In [ ]:
import os

MARCA = "/content/.entorno_listo"
REPO = "https://github.com/Seloga26/ADA2026_2"
RAMA = "main"  # cambiar a la rama del PR mientras coordenadas no esté en main

if not os.path.exists(MARCA):
    !pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate nltk spacy emoji langdetect immutabledict
    !python -m spacy download en_core_web_sm -q
    !git clone -q https://github.com/allenai/open-instruct /content/open-instruct
    !git clone -q -b {RAMA} {REPO} /content/ADA2026_2

    import io
    import urllib.request
    import zipfile

    NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
    PAQUETES = [
        ("tokenizers", "punkt"),
        ("tokenizers", "punkt_tab"),
        ("taggers", "averaged_perceptron_tagger"),
        ("taggers", "averaged_perceptron_tagger_eng"),
    ]
    for carpeta, nombre in PAQUETES:
        with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
            zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

    open(MARCA, "w").close()
    print("Entorno instalado. El entorno se reinicia: vuelva a ejecutar todas las celdas.")
    os.kill(os.getpid(), 9)

# Si el entorno ya existía (por otro notebook), asegura la rama correcta.
!cd /content/ADA2026_2 && git fetch -q origin && git checkout -q {RAMA} && git pull -q origin {RAMA}
%cd /content/ADA2026_2
print("Entorno listo")

## Parámetros (D17)

`B = 14 400` generaciones: dos arranques de hasta 2 pasadas. El caché y el historial viven en Drive.
Cada arranque se corre por separado (`ARRANQUE = 0`, luego `1`) para que una desconexión
solo cueste ese arranque.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
CARPETA = "/content/drive/MyDrive/oraculo_ada"
os.makedirs(CARPETA, exist_ok=True)

ARRANQUE = 0          # 0 en la primera corrida, 1 en la segunda
MAX_PASADAS = 2
B = 14_400            # D17
SEMILLA = 1
SEMILLA_BUSQUEDA = 0
RUTA_CACHE = f"{CARPETA}/oraculo_cache.json"
RUTA_HISTORIAL = f"{CARPETA}/historial_arranque_{ARRANQUE}.csv"

In [ ]:
import json

from ayudas import cargar_datos, cargar_modelo, dividir
from busqueda_prompt.algoritmos import coordenadas
from busqueda_prompt.contratos import OPCIONES_POR_RANURA, RANURAS
from oraculo import Oraculo

modelo = cargar_modelo("qwen17b")
busqueda, validacion = dividir(cargar_datos())
oraculo = Oraculo(modelo, busqueda, validacion, cache=RUTA_CACHE, lote=16, presupuesto=32_000_000)
print(len(busqueda), "instancias de búsqueda;", len(validacion), "de validación")

## Búsqueda

Un solo arranque por corrida: `semilla_busqueda` y el índice de arranque fijan el punto de
partida, así que repetir la celda reproduce el mismo recorrido (lo ya generado sale del caché).
Cada fila del historial se guarda en Drive al instante.

In [ ]:
# Para reanudar tras una desconexión: borre el CSV de este arranque (los pasos se repiten
# desde el caché sin costo de GPU) y vuelva a ejecutar esta celda.
historial = []
resultado = coordenadas.resolver(
    oraculo, RANURAS, (OPCIONES_POR_RANURA,) * len(RANURAS), busqueda,
    semilla=SEMILLA, presupuesto=B, arranques=1, max_pasadas=MAX_PASADAS,
    semilla_busqueda=SEMILLA_BUSQUEDA * 1000 + ARRANQUE,
    historial=historial, ruta_historial=RUTA_HISTORIAL,
    mostrar=lambda f: print(f"{f['paso']:4d} {f['ranura']:>10} {f['opcion']!s:>2} "
                            f"{f['precision']:6.1%} mejor {f['mejor_precision']:6.1%} "
                            f"gen {f['consultas']:6d} {f['decision']}"),
)
print("\nGanadora del arranque:", json.dumps(resultado.ganadora))
print(f"Precisión: {resultado.precision:.1%}   generaciones nuevas: {resultado.consultas}")

In [ ]:
# Resumen para anotar en la bitácora y comparar con el otro arranque.
resumen = {"arranque": ARRANQUE, "config": resultado.ganadora,
           "precision": resultado.precision, "consultas": resultado.consultas}
with open(f"{CARPETA}/resumen_arranque_{ARRANQUE}.json", "w", encoding="utf-8") as f:
    json.dump(resumen, f, indent=1)
print(resumen)